## 1. Preparar el entorno

Una sesión de Python compartida para todo el experimento.

In [ ]:
# Importaciones necesarias
import random
import time
import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

RUTA_RESULTADOS = "resultados"

# Crear la carpeta de resultados si no existe
os.makedirs(RUTA_RESULTADOS, exist_ok=True)

# Subcarpeta para gráficas individuales
RUTA_GRAFICAS = os.path.join(RUTA_RESULTADOS, "graficas")
os.makedirs(RUTA_GRAFICAS, exist_ok=True)

print(f"Los resultados se guardarán en: {os.path.abspath(RUTA_RESULTADOS)}")

print(f"Python: {sys.version.split()[0]}")
print(f"CPU lógicas visibles: {os.cpu_count()}")
if hasattr(os, 'sched_getaffinity'):
    print(f"CPU permitidas por afinidad: {len(os.sched_getaffinity(0))}")
from pathlib import Path
quota = Path('/sys/fs/cgroup/cpu.max')
if quota.exists():
    print(f"Cuota cgroup (cuota / período): {quota.read_text().strip()}")
print("Entorno listo. La cuota de Docker puede limitar los recursos efectivos.")
plt.rcParams.update({'figure.facecolor':'#FFF9F0', 'axes.facecolor':'#FFFFFF',
    'axes.prop_cycle':plt.cycler(color=['#6C63FF','#FF6584','#2CB67D']), 'font.size':13})

## 2. El azar aproxima π

Lanzamos 1,000 puntos. La proporción dentro del círculo permite estimar π.

In [ ]:
# Generar pocos puntos para visualizar el método
n = 1000
dentro_x, dentro_y = [], []
fuera_x, fuera_y = [], []

for _ in range(n):
    x = random.uniform(-1, 1)
    y = random.uniform(-1, 1)
    if x*x + y*y <= 1:
        dentro_x.append(x)
        dentro_y.append(y)
    else:
        fuera_x.append(x)
        fuera_y.append(y)

pi_estimado = 4 * len(dentro_x) / n
print(f"Con {n} puntos, π estimado = {pi_estimado:.4f}")

plt.figure(figsize=(6,4.6))
plt.scatter(dentro_x, dentro_y, color='#6C63FF', s=5, label='Dentro')
plt.scatter(fuera_x, fuera_y, color='#FF6584', s=5, label='Fuera')
circulo = plt.Circle((0,0), 1, fill=False, color='blue', linewidth=2)
plt.gca().add_patch(circulo)
plt.axis('equal')
plt.title(f'Monte Carlo para π (n={n})')
plt.legend()
plt.show()

## 3. Un solo proceso

Definimos la referencia secuencial y probamos con un millón de puntos.

In [ ]:
def montecarlo_secuencial(n_puntos):
    """
    Estima π usando simulación Monte Carlo de forma secuencial.
    
    Parámetros:
    n_puntos (int): Número total de puntos a generar.
    
    Retorna:
    float: Estimación de π.
    """
    dentro_circulo = 0
    for _ in range(n_puntos):
        x = random.uniform(-1, 1)
        y = random.uniform(-1, 1)
        if x*x + y*y <= 1:
            dentro_circulo += 1
    return 4 * dentro_circulo / n_puntos

n_prueba = 1_000_000
inicio = time.perf_counter()
pi_estimado = montecarlo_secuencial(n_prueba)
fin = time.perf_counter()

print(f"Con {n_prueba:,} puntos:")
print(f"π estimado = {pi_estimado:.6f}")
print(f"Tiempo = {fin - inicio:.4f} segundos")
print(f"Error absoluto = {abs(pi_estimado - 3.141592653589793):.6f}")

## 4. Varios procesos

Cada trabajador cuenta su parte. El proceso principal reúne los conteos.

In [ ]:
import montecarlo_worker as mw

def montecarlo_paralelo(n_puntos, n_procesos):
    """
    Estima π usando simulación Monte Carlo con múltiples procesos.
    
    Parámetros:
    n_puntos (int): Número total de puntos.
    n_procesos (int): Número de procesos a utilizar.
    
    Retorna:
    float: Estimación de π.
    """
    # Dividir el trabajo en chunks
    chunk_size = n_puntos // n_procesos
    resto = n_puntos % n_procesos
    chunks = [chunk_size] * n_procesos
    for i in range(resto):
        chunks[i] += 1
    
    # Ejecutar en paralelo
    with ProcessPoolExecutor(max_workers=n_procesos) as executor:
        resultados = list(executor.map(mw.contar_puntos_en_circulo, chunks))
    
    total_dentro = sum(resultados)
    return 4 * total_dentro / n_puntos

n_prueba = 1_000_000
n_procesos = 4
inicio = time.perf_counter()
pi_estimado = montecarlo_paralelo(n_prueba, n_procesos)
fin = time.perf_counter()

print(f"Con {n_prueba:,} puntos y {n_procesos} procesos:")
print(f"π estimado = {pi_estimado:.6f}")
print(f"Tiempo = {fin - inicio:.4f} segundos")
print(f"Error absoluto = {abs(pi_estimado - 3.141592653589793):.6f}")

## 5. Diseñar la comparación

Puedes editar N_PUNTOS y REPETICIONES antes de ejecutar esta celda.

In [ ]:
from laboratorio import medir
N_PUNTOS = 10_000_000
REPETICIONES = 3
PROCESOS = [1, 2, 4, 8]
assert isinstance(N_PUNTOS, int) and N_PUNTOS > 0
assert isinstance(REPETICIONES, int) and REPETICIONES >= 2
muestras = {}
print(f"{N_PUNTOS:,} puntos por corrida")
print(f"{REPETICIONES} repeticiones por configuración")
print("Ejecutaremos cada configuración por separado.")

## 6. Medir la referencia

El promedio secuencial será la base del speedup.

In [ ]:
muestras['Secuencial'] = medir(
    montecarlo_secuencial, N_PUNTOS, None, REPETICIONES
)

## 7. Medir con 1 proceso

La medición incluye crear el pool, calcular y reunir los resultados.

In [ ]:
muestras['Paralelo 1 proc'] = medir(
    montecarlo_paralelo, N_PUNTOS, 1, REPETICIONES
)

## 8. Medir con 2 procesos

La medición incluye crear el pool, calcular y reunir los resultados.

In [ ]:
muestras['Paralelo 2 proc'] = medir(
    montecarlo_paralelo, N_PUNTOS, 2, REPETICIONES
)

## 9. Medir con 4 procesos

La medición incluye crear el pool, calcular y reunir los resultados.

In [ ]:
muestras['Paralelo 4 proc'] = medir(
    montecarlo_paralelo, N_PUNTOS, 4, REPETICIONES
)

## 10. Medir con 8 procesos

La medición incluye crear el pool, calcular y reunir los resultados.

In [ ]:
muestras['Paralelo 8 proc'] = medir(
    montecarlo_paralelo, N_PUNTOS, 8, REPETICIONES
)

## 11. Comparar los resultados

Tiempo promedio, dispersión, aceleración y eficiencia de esta sesión.

In [ ]:
df_resultados = pd.DataFrame([
    fila for grupo in muestras.values() for fila in grupo
])
resumen = df_resultados.groupby(['configuracion', 'procesos'], sort=False).agg(
    tiempo_promedio=('tiempo', 'mean'),
    tiempo_std=('tiempo', 'std'),
    pi_promedio=('pi', 'mean')
).reset_index()
tiempo_secuencial = resumen.loc[
    resumen['configuracion'] == 'Secuencial', 'tiempo_promedio'
].iloc[0]
resumen['speedup'] = tiempo_secuencial / resumen['tiempo_promedio']
resumen['eficiencia'] = resumen['speedup'] / resumen['procesos']
display(resumen[['configuracion', 'tiempo_promedio', 'tiempo_std',
                 'speedup', 'eficiencia']].round(4))
# Copias en el directorio de resultados del contenedor.
df_resultados.to_csv('resultados/resultados_completos.csv', index=False)
resumen.to_csv('resultados/resumen.csv', index=False)

## 12. Tiempo de ejecución

Menos segundos significa una ejecución más rápida. Las barras incluyen la desviación estándar.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(resumen['configuracion'], resumen['tiempo_promedio'],
       yerr=resumen['tiempo_std'], capsize=5, color='#6C63FF')
ax.set_ylabel('Tiempo promedio (s)')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## 13. Aceleración real e ideal

Speedup = tiempo secuencial / tiempo paralelo.

In [ ]:
paralelos = resumen[resumen['configuracion'] != 'Secuencial']
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(paralelos['procesos'], paralelos['speedup'], 'o-',
        color='#6C63FF', linewidth=3, label='Medido')
ax.plot(paralelos['procesos'], paralelos['procesos'], '--',
        color='#FF6584', label='Ideal lineal')
ax.set(xlabel='Procesos', ylabel='Speedup', xticks=[1,2,4,8])
ax.legend()
plt.tight_layout()
plt.show()

## 14. Eficiencia por proceso

Eficiencia = speedup / procesos. Terminar antes no implica mantener la misma eficiencia.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(paralelos['procesos'].astype(str),
       paralelos['eficiencia'] * 100, color='#2CB67D')
ax.axhline(100, linestyle='--', color='#FF6584', label='Ideal 100 %')
ax.set(xlabel='Procesos', ylabel='Eficiencia (%)')
ax.legend()
plt.tight_layout()
plt.show()

## 15. Interpretar esta corrida

La conclusión depende de los resultados, no de una aceleración prometida.

In [ ]:
mejor = resumen.loc[resumen['tiempo_promedio'].idxmin()]
print(f"Menor tiempo: {mejor['configuracion']}")
print(f"Promedio: {mejor['tiempo_promedio']:.4f} s")
print(f"Speedup: {mejor['speedup']:.2f}×")
print(f"Eficiencia: {mejor['eficiencia']:.1%}")
print("\nMás procesos no garantizan mejor rendimiento.")
print("Esta sesión demuestra paralelismo en un solo equipo.")
print("Un clúster añade comunicación y coordinación entre nodos.")